# 🛡️ ContractGuard — Legal Contract Clause Risk & Obligation Detection
**NLP CA3: Build–Audit–Pitch · Group 44** (Aarav Seth, Aryan Bakshi, Rudranil Ghosh)

Dataset: **CUAD v1** (Contract Understanding Atticus Dataset, The Atticus Project, CC BY 4.0) — 510 real contracts, 41 expert-labelled clause types.

| Phase | What we do |
|---|---|
| **BUILD** | TF-IDF + Logistic Regression classifies a clause into 14 risk-relevant types (+ Neutral); a rule engine maps to risk tier and extracts obligations |
| **AUDIT** | Compare performance across clause length and contract type; focus on *high-risk recall* |
| **PITCH** | Streamlit app: paste a clause or scan a whole contract |

> Run all cells top to bottom.

## 0. Setup
**In Colab:** upload this project folder (or `git clone` your repo) so that `src/` exists in the working directory, then run the cell. **Locally:** run from the project root.

In [ ]:
import os, sys, subprocess
# --- Colab only: uncomment and put your repo URL ---
# !git clone https://github.com/<your-username>/contractguard.git
# %cd contractguard
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')            # opened from notebooks/ -> go to project root
if not os.path.exists('src/prepare_data.py'):
    raise SystemExit('Folder not found: make sure src/ is in the current directory (see markdown above).')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lime', 'seaborn', 'scikit-learn', 'pandas', 'matplotlib', 'requests', 'joblib'])
sys.path.insert(0, 'src')
print('Ready')

## Step 1 — Prepare the data
Downloads CUAD, extracts the labelled clause spans for 14 clause types, and samples neutral contract text that does *not* overlap any of them.

In [ ]:
!python src/prepare_data.py

### 1.1 Explore the data (put these plots in your report)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
df = pd.read_csv('data/clauses.csv')
print(df.shape, '| contracts:', df.doc.nunique())
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
df.label.value_counts().sort_values().plot.barh(ax=ax[0], color='#2b6cb0'); ax[0].set_title('Clauses per type')
sns.histplot(df[df.n_words<250].n_words, bins=40, ax=ax[1]); ax[1].set_title('Clause length (words)')
plt.tight_layout(); plt.show()
df.groupby('label').n_words.median().sort_values()

In [ ]:
df.sample(5, random_state=1)[['label','text']]

## Step 2 — Build + Audit + Explain
Splits **by contract** (no leakage), compares 3 models, picks Logistic Regression (explainable), runs the subgroup audit and saves all charts/tables to `outputs/`.

In [ ]:
!python src/train_audit.py

### 2.1 Results

In [ ]:
import json
print(json.dumps(json.load(open('outputs/metrics.json')), indent=2))
pd.read_csv('outputs/model_comparison.csv')

In [ ]:
pd.read_csv('outputs/per_class_report.csv', index_col=0).sort_values('f1-score')

In [ ]:
from IPython.display import Image, display
display(Image('outputs/confusion_matrix.png'))

### 2.2 AUDIT — does the model work equally well everywhere?

In [ ]:
print('By clause length'); display(pd.read_csv('outputs/audit_by_length.csv'))
print('By contract type');  display(pd.read_csv('outputs/audit_by_doctype.csv'))
display(Image('outputs/audit_charts.png'))

**Write your audit findings here** (2–3 sentences): which subgroup is weakest? By how many points? Is the gap *bigger for high-risk clauses*? Your hypothesis for why (e.g. long clauses mix several topics; Distribution contracts use different vocabulary)? What would you do about it?

### 2.3 EXPLAINABILITY — what words drive predictions?

In [ ]:
display(Image('outputs/top_features.png'))

In [ ]:
import joblib, numpy as np
from lime.lime_text import LimeTextExplainer
pipe = joblib.load('models/contractguard.joblib')
exp = LimeTextExplainer(class_names=list(pipe.named_steps['clf'].classes_))
txt = "During the Term and for two (2) years thereafter, Distributor shall not market or sell any product that competes with the Products."
e = exp.explain_instance(txt, pipe.predict_proba, num_features=8, top_labels=1)
label = e.available_labels()[0]
w = pd.DataFrame(e.as_list(label=label), columns=['word', 'weight']).sort_values('weight')
plt.figure(figsize=(7, 4)); plt.barh(w.word, w.weight, color=['#c53030' if x < 0 else '#2b6cb0' for x in w.weight])
plt.title('LIME: words pushing toward / away from "' + e.class_names[label] + '"'); plt.tight_layout(); plt.show()

### 2.4 Error analysis — the most confident mistakes

In [ ]:
pd.read_csv('outputs/error_analysis_top40.csv').head(10)

**Write your error-analysis notes here:** pick 3 mistakes and say *why* the model confused them (e.g. Uncapped vs Capped liability share almost the same vocabulary — the difference is one word like 'except' or 'unlimited').

## Step 3 — Obligation detection (rule-based)
Transparent regex engine: OBLIGATION (shall/must/will), PROHIBITION (shall not/may not), RIGHT (may/entitled to), plus deadlines, money amounts and the acting party.

In [ ]:
from risk_rules import extract_obligations, RISK_MAP, PLAIN_ENGLISH
demo = ("Supplier shall deliver the Products within thirty (30) days of each purchase order. "
        "Distributor may not sell competing products. "
        "Either party shall have the right to terminate this Agreement upon 60 days notice. "
        "Buyer will pay $50,000 per year.")
pd.DataFrame(extract_obligations(demo))

### 3.1 (Optional, strong for marks) validate the rules by hand
Label the 60 sentences below yourselves (`OBLIGATION / PROHIBITION / RIGHT / NONE`) in the `human_label` column and compare with the rules. Report the accuracy honestly.

In [ ]:
import random
from risk_rules import split_sentences, classify_sentence
random.seed(7)
sents = [s for t in df.text.sample(400, random_state=3) for s in split_sentences(t)]
sample = random.sample(sents, 60)
out = pd.DataFrame({'sentence': sample, 'rule_label': [classify_sentence(s) for s in sample], 'human_label': ''})
out.to_csv('outputs/obligation_validation_template.csv', index=False)
out.head()

## Step 4 — Run the demo app
In a terminal at the project root:
```
pip install -r requirements.txt
streamlit run app.py
```
(Colab cannot open the app window directly — record the demo on a laptop.)

## Step 5 — Limitations (copy into the report)
* CUAD is **US SEC-filed commercial contracts**; Indian contracts / other drafting styles may behave differently.
* Risk tiers are **our assumption**, from a small-signer viewpoint, not legal truth.
* The Neutral class assumes un-annotated text is non-risky; annotation gaps are possible.
* Bag-of-words cannot read negation/scope well (e.g. "liability is **not** capped").
* **Not legal advice.**